# 🌲 SmartForest – Forest Fire Risk Prediction
## Notebook 3: Visualization + Model Development
**SRM Institute of Science and Technology | AIML-B | 2026**

---
### Steps covered:
1. Correlation Heatmap
2. Feature Distribution by Class
3. Monthly Fire Frequency
4. Temperature vs FWI Scatter
5. Model Training (LR, DT, RF)
6. Confusion Matrices
7. Model Comparison
8. Feature Importance


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report)

sns.set_theme(style='darkgrid', palette='muted')
os.makedirs('../outputs/graphs', exist_ok=True)
os.makedirs('../outputs/results', exist_ok=True)
print('Libraries loaded ✅')

In [ ]:
df = pd.read_csv('../dataset/processed_data/cleaned_forest_fires.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/correlation_heatmap.png', dpi=150)
plt.show()

## 2. Environmental Feature Distributions by Class

In [ ]:
features = ['Temperature', 'RH', 'Ws', 'Rain', 'FWI', 'FFMC']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.boxplot(x='Classes', y=feat, data=df, ax=axes[i],
                palette={0: '#2E8B57', 1: '#E05C2A'})
    axes[i].set_title(f'{feat} vs Fire Class', fontweight='bold')
    axes[i].set_xlabel('0 = Not Fire | 1 = Fire')

plt.suptitle('Environmental Features vs Fire Risk', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/feature_distributions.png', dpi=150)
plt.show()

## 3. Monthly Fire Frequency

In [ ]:
fire_df = df[df['Classes'] == 1]
month_counts = fire_df['month'].value_counts().sort_index()
month_names = {6: 'June', 7: 'July', 8: 'August', 9: 'September'}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([month_names.get(m, str(m)) for m in month_counts.index],
       month_counts.values, color='#E05C2A', edgecolor='white')
ax.set_title('Monthly Fire Frequency', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Fire Incidents')
plt.tight_layout()
plt.savefig('../outputs/graphs/monthly_fire_frequency.png', dpi=150)
plt.show()

## 4. Temperature vs FWI (Fire Weather Index)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = df['Classes'].map({0: '#2E8B57', 1: '#E05C2A'})
ax.scatter(df['Temperature'], df['FWI'], c=colors, alpha=0.6, edgecolors='white', linewidth=0.3)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('FWI – Fire Weather Index', fontsize=12)
ax.set_title('Temperature vs FWI by Fire Class', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend = [Patch(color='#E05C2A', label='Fire'), Patch(color='#2E8B57', label='Not Fire')]
ax.legend(handles=legend)
plt.tight_layout()
plt.savefig('../outputs/graphs/temp_vs_fwi.png', dpi=150)
plt.show()

## 5. Model Training

In [ ]:
feature_cols = ['Temperature', 'RH', 'Ws', 'Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']
X = df[feature_cols]
y = df['Classes']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

trained, results = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    y_pred = model.predict(X_test)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred) * 100, 2)
    }
    print(f'\n{name}:')
    print(classification_report(y_test, y_pred, target_names=['Not Fire', 'Fire']))

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='YlOrRd',
                xticklabels=['Not Fire', 'Fire'], yticklabels=['Not Fire', 'Fire'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices – All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/confusion_matrices.png', dpi=150)
plt.show()

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)
results_df.to_csv('../outputs/results/model_results.csv')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(results.keys())
x = np.arange(len(metrics))
width = 0.25
colors_m = ['#3B82F6', '#10B981', '#F59E0B']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, color) in enumerate(zip(model_names, colors_m)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 115)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/graphs/model_comparison.png', dpi=150)
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
rf = trained['Random Forest']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(feature_cols)), importances[indices], color='#E05C2A', edgecolor='white')
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels([feature_cols[i] for i in indices], rotation=30, ha='right')
ax.set_title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/graphs/feature_importance.png', dpi=150)
plt.show()

best = max(results, key=lambda k: results[k]['Accuracy'])
print(f'\n✅ Best Model: {best} — {results[best]["Accuracy"]}% Accuracy')
print('All outputs saved to outputs/graphs/ and outputs/results/')